# MLA Phase 5 — XLM-R: Đánh giá một lần trên Test (Inference-only)

Notebook này **không huấn luyện lại**, không chọn epoch/siêu tham số và không thay đổi checkpoint. Nó nạp `outputs/best_model` (checkpoint đã chọn bằng Dev PR-AUC, epoch 2) rồi chạy dự đoán đúng một lần trên Test đã khóa.

**Điều kiện trước khi chạy**
- Đã chạy xong notebook Train/Dev và có `MLA_Phase5_XLMR/outputs/best_model/`.
- Đưa `test_segments.csv` chính thức vào `MLA_Phase5_XLMR/data/`.
- Không dùng kết quả Test để điều chỉnh mô hình, ngưỡng, đặc trưng hoặc lựa chọn checkpoint.
- Sau lần đánh giá này, Test được xem là đã mở; mọi thay đổi mô hình tiếp theo cần được báo cáo là nghiên cứu mới, không được tiếp tục dùng Test như tập giữ kín.

## 0. Cài đặt và chọn GPU

Trong Colab chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU khả dụng). Chạy lần lượt từ trên xuống.

In [1]:
!pip -q install "transformers==4.46.3" "accelerate==1.1.1" "scikit-learn>=1.5,<2" "sentencepiece>=0.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [9]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

# Tìm checkpoint-850 trong MyDrive
MYDRIVE = Path("/content/drive/MyDrive")

matches = list(MYDRIVE.rglob("checkpoint-850"))

print("Số checkpoint-850 tìm thấy:", len(matches))

for p in matches:
    print("\nĐường dẫn:", p)
    print("Là thư mục:", p.is_dir())

    if p.is_dir():
        print("Các tệp bên trong:")
        for f in sorted(p.iterdir()):
            print(" -", f.name)

Mounted at /content/drive
Số checkpoint-850 tìm thấy: 1

Đường dẫn: /content/drive/MyDrive/MLA_Phase5_XLMR/outputs/checkpoints/checkpoint-850
Là thư mục: True
Các tệp bên trong:
 - config.json
 - model.safetensors
 - optimizer.pt
 - rng_state.pth
 - scheduler.pt
 - sentencepiece.bpe.model
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - trainer_state.json
 - training_args.bin


In [10]:
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/MLA_Phase5_XLMR/outputs")

print("Các thư mục và tệp trong outputs:")
for item in sorted(OUTPUT_DIR.iterdir()):
    print(item.name, "[DIR]" if item.is_dir() else "[FILE]")

print("\nCác checkpoint:")
for item in sorted(OUTPUT_DIR.glob("checkpoint-*")):
    print(item)

Các thư mục và tệp trong outputs:
checkpoints [DIR]

Các checkpoint:


In [11]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/MLA_Phase5_XLMR")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

# Checkpoint tốt nhất được lựa chọn trên Dev
MODEL_DIR = OUTPUT_DIR / "checkpoints" / "checkpoint-850"

# Tập Test chính thức
TEST_PATH = DATA_DIR / "test_segments.csv"

assert MODEL_DIR.is_dir(), f"Không tìm thấy checkpoint: {MODEL_DIR}"
assert (MODEL_DIR / "config.json").is_file(), "Thiếu config.json"
assert (
    (MODEL_DIR / "model.safetensors").is_file()
    or (MODEL_DIR / "pytorch_model.bin").is_file()
), "Không tìm thấy trọng số mô hình"

assert TEST_PATH.is_file(), f"Không tìm thấy Test: {TEST_PATH}"

print("Checkpoint:", MODEL_DIR)
print("Test data:", TEST_PATH)
print("Đã xác minh các tệp cần thiết.")

Checkpoint: /content/drive/MyDrive/MLA_Phase5_XLMR/outputs/checkpoints/checkpoint-850
Test data: /content/drive/MyDrive/MLA_Phase5_XLMR/data/test_segments.csv
Đã xác minh các tệp cần thiết.


In [12]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

BASE_MODEL = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
)

print("Đã tải thành công XLM-R checkpoint-850.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Đã tải thành công XLM-R checkpoint-850.


In [13]:
import json, time, platform
import numpy as np
import pandas as pd
import torch, transformers, sklearn
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from torch.utils.data import Dataset, DataLoader

SEED = 42
MAX_LENGTH = 256
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("Python:", platform.python_version(), "| torch:", torch.__version__,
      "| transformers:", transformers.__version__, "| sklearn:", sklearn.__version__)

Device: cuda
Tesla T4
Python: 3.13.16 | torch: 2.14.1+cu130 | transformers: 4.46.3 | sklearn: 1.6.1


## 1. Nạp và xác minh Test

Chỉ đọc `test_segments.csv` từ split chính thức. Các kiểm tra số lượng dưới đây khớp với split Phase 1 đã khóa: 1.575 mẫu, 162 nhãn dương, 20 documents. Nếu assertion thất bại, **dừng lại**, không tự sửa hoặc tạo lại split.

In [14]:
REQUIRED = {"system","doc","doc_id","seg_id","source","target","y_primary"}
test_df = pd.read_csv(TEST_PATH)
assert REQUIRED.issubset(test_df.columns), f"Thiếu cột: {REQUIRED-set(test_df.columns)}"
assert test_df["y_primary"].notna().all()
assert set(test_df["y_primary"].unique()).issubset({0,1})
assert test_df[["source","target"]].notna().all().all()
assert (test_df["source"].astype(str).str.strip() != "").all()
assert (test_df["target"].astype(str).str.strip() != "").all()
assert len(test_df) == 1575, f"Sai số mẫu Test: {len(test_df)}"
assert int(test_df.y_primary.sum()) == 162, f"Sai số nhãn dương: {int(test_df.y_primary.sum())}"
assert test_df.doc.nunique() == 20, f"Sai số documents: {test_df.doc.nunique()}"
assert not test_df.duplicated(["system","doc_id","seg_id"]).any(), "Trùng khóa segment-system"
print("PASS | Test rows:", len(test_df), "| positives:", int(test_df.y_primary.sum()),
      "| prevalence:", round(test_df.y_primary.mean(), 6), "| docs:", test_df.doc.nunique())

PASS | Test rows: 1575 | positives: 162 | prevalence: 0.102857 | docs: 20


## 2. Nạp đúng checkpoint epoch 2 và tokenizer

Không tải model pretrained mới để huấn luyện; chỉ nạp artifact đã lưu trong `best_model/`.

In [15]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.to(DEVICE)
model.eval()
print("Loaded model:", MODEL_DIR)

Loaded model: /content/drive/MyDrive/MLA_Phase5_XLMR/outputs/checkpoints/checkpoint-850


## 3. Dự đoán Test (không huấn luyện)

Giữ nguyên thứ tự dòng của CSV để gắn score với đúng segment. Không thay đổi ngưỡng hoặc cấu hình sau khi xem kết quả.

In [16]:
class TestPairDataset(Dataset):
    def __init__(self, frame):
        self.enc = tokenizer(
            frame.source.astype(str).tolist(),
            frame.target.astype(str).tolist(),
            truncation=True, max_length=MAX_LENGTH, padding=False
        )
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}

collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
test_ds = TestPairDataset(test_df)
loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collator)

all_probs = []
start = time.time()
with torch.inference_mode():
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy().tolist())
inference_seconds = time.time() - start
test_scores = np.asarray(all_probs, dtype=float)
assert len(test_scores) == len(test_df) and np.isfinite(test_scores).all()
print("Inference seconds:", round(inference_seconds, 2), "| predictions:", len(test_scores))

Inference seconds: 14.04 | predictions: 1575


## 4. Tính chỉ số một lần trên Test

Báo cáo AP (Average Precision; dự án gọi là PR-AUC theo quy ước), ROC-AUC và các chỉ số ưu tiên rà soát top-20%. `k = ceil(0.20 × N)`. MCC@top20 được tính giữa nhãn thật và vector chọn/không chọn top-20%. Không chọn mô hình dựa trên các kết quả này.

In [17]:
def top20_metrics(y_true, scores):
    y_true = np.asarray(y_true, dtype=int)
    n = len(y_true)
    k = int(np.ceil(0.20 * n))
    order = np.argsort(-np.asarray(scores), kind="mergesort")
    chosen = order[:k]
    selected = y_true[chosen]
    positives = int(y_true.sum())
    tp = int(selected.sum())
    precision = tp / k
    recall = tp / positives if positives else 0.0
    prevalence = positives / n
    selected_mask = np.zeros(n, dtype=int)
    selected_mask[chosen] = 1
    return {
        "k_top20": k,
        "true_positives_top20": tp,
        "precision_at_20": precision,
        "recall_at_20": recall,
        "lift_at_20": precision / prevalence if prevalence else 0.0,
        "mcc_top20": float(matthews_corrcoef(y_true, selected_mask))
    }

y_test = test_df.y_primary.astype(int).to_numpy()
test_metrics = {
    "model": "FacebookAI/xlm-roberta-base",
    "checkpoint": "outputs/best_model (selected by Dev PR-AUC; epoch 2)",
    "split": "canonical_test",
    "n_test": int(len(y_test)),
    "positives": int(y_test.sum()),
    "prevalence": float(y_test.mean()),
    "average_precision_pr_auc": float(average_precision_score(y_test, test_scores)),
    "roc_auc": float(roc_auc_score(y_test, test_scores)),
    **top20_metrics(y_test, test_scores),
    "decision_threshold_0_5_metrics_not_used_for_model_selection": {
        "precision": float(precision_score(y_test, (test_scores >= .5).astype(int), zero_division=0)),
        "recall": float(recall_score(y_test, (test_scores >= .5).astype(int), zero_division=0)),
        "mcc": float(matthews_corrcoef(y_test, (test_scores >= .5).astype(int)))
    },
    "inference_seconds": float(inference_seconds),
    "test_used_for_training_or_selection": False
}
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))

{
  "model": "FacebookAI/xlm-roberta-base",
  "checkpoint": "outputs/best_model (selected by Dev PR-AUC; epoch 2)",
  "split": "canonical_test",
  "n_test": 1575,
  "positives": 162,
  "prevalence": 0.10285714285714286,
  "average_precision_pr_auc": 0.1495860225495732,
  "roc_auc": 0.6482944964308494,
  "k_top20": 315,
  "true_positives_top20": 64,
  "precision_at_20": 0.20317460317460317,
  "recall_at_20": 0.3950617283950617,
  "lift_at_20": 1.9753086419753088,
  "mcc_top20": 0.16511955255801825,
  "decision_threshold_0_5_metrics_not_used_for_model_selection": {
    "precision": 0.10204081632653061,
    "recall": 0.09259259259259259,
    "mcc": -0.0008622048170500978
  },
  "inference_seconds": 14.036864280700684,
  "test_used_for_training_or_selection": false
}


## 5. Lưu dự đoán, metrics và biên bản chạy

Các file được lưu vào `MyDrive/MLA_Phase5_XLMR/outputs/`. Không ghi đè kết quả Train/Dev.

In [18]:
pred_cols = [c for c in ["system","doc","doc_id","seg_id","source","target","y_primary"] if c in test_df.columns]
test_pred = test_df[pred_cols].copy()
test_pred["score_XLMR"] = test_scores
test_pred["prediction_at_0_5"] = (test_scores >= 0.5).astype(int)
pred_path = OUTPUT_DIR / "test_predictions_xlmr.csv"
metrics_path = OUTPUT_DIR / "test_metrics_xlmr.json"
run_path = OUTPUT_DIR / "test_inference_run.json"
test_pred.to_csv(pred_path, index=False)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2, ensure_ascii=False)
run_info = {
    "model_dir": str(MODEL_DIR),
    "model": "FacebookAI/xlm-roberta-base",
    "max_length": MAX_LENGTH,
    "batch_size": 16,
    "seed_for_reproducibility": SEED,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "test_rows": len(test_df),
    "prediction_file": pred_path.name,
    "metrics_file": metrics_path.name,
    "training_performed": False,
    "checkpoint_or_hyperparameter_selection_on_test": False
}
with open(run_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2, ensure_ascii=False)
assert len(test_pred) == 1575 and test_pred.score_XLMR.notna().all()
print("Saved:", pred_path)
print("Saved:", metrics_path)
print("Saved:", run_path)

Saved: /content/drive/MyDrive/MLA_Phase5_XLMR/outputs/test_predictions_xlmr.csv
Saved: /content/drive/MyDrive/MLA_Phase5_XLMR/outputs/test_metrics_xlmr.json
Saved: /content/drive/MyDrive/MLA_Phase5_XLMR/outputs/test_inference_run.json


## 6. Kết thúc và nguyên tắc báo cáo

- Đây là **đánh giá Test một lần** bằng checkpoint đã chọn trên Dev.
- Không chạy thêm epoch, không đổi checkpoint, không điều chỉnh ngưỡng hoặc feature dựa trên Test.
- Trong final report, trình bày Test như kết quả đánh giá cuối cùng; phân biệt rõ kết quả Dev dùng chọn checkpoint và kết quả Test dùng ước lượng hiệu năng cuối.
- Nếu có lỗi assertion ở bước 1, dừng và kiểm tra đúng file split; không tự tạo split mới.